# Figure 5 — public-source ecological analysis

This notebook reconstructs every Figure 5 plotting table directly from the
public files acquired by `00_Paper_Data_Download.ipynb`:

- GSE159115 single-cell 10x matrices and ccRCC cell annotation;
- JAVELIN Renal 101 clinical and normalized RNA-seq supplementary tables;
- CPTAC ccRCC LinkedOmics tumor RNA-seq;
- GSE210041 Visium matrices and spatial coordinates.

There are no archived effects, P values, confidence intervals, or expected
results in this notebook. Checks are restricted to public-source integrity,
data structure, analyzable sample counts, and the file contract used by the
unchanged plotting notebook.


In [1]:
from pathlib import Path
import hashlib
import os
import re
import tarfile
import warnings

import h5py
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import pearsonr, spearmanr, wilcoxon
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_5_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MITOXY_GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
MYELOID_GENES = ["MYD88", "TNF", "TLR2", "TLR4"]
SINGLE_CELL_FAMILY = [
    "BAX", "BAK1", "BID", "SLC7A11", "MYD88", "TNF",
    "TLR2", "TLR4", "RICTOR", "MTOR", "MAPKAP1",
]
ANGIO_GENES = ["VEGFA", "KDR", "ESM1", "PECAM1", "ANGPTL4", "CD34"]
TUMOR_MARKERS = ["CA9", "NDUFA4L2", "EGLN3", "KRT8", "KRT18", "EPCAM"]
MYELOID_MARKERS = ["LST1", "TYROBP", "FCER1G", "CTSS", "AIF1", "CD68"]

manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset)
        & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); "
            f"found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "single_cell_tar": source_path("GSE159115", "GSE159115_RAW.tar"),
    "single_cell_annotation": source_path(
        "GSE159115", "GSE159115_ccRCC_anno.csv.gz"
    ),
    "spatial_tar": source_path("GSE210041", "GSE210041_RAW.tar"),
    "javelin_workbook": source_path(
        "JAVELIN Renal 101", "clinical and molecular tables"
    ),
    "cptac_rna": source_path("CPTAC ccRCC", "tumor RNA expression"),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


,analysis_key,path
0,single_cell_tar,/scratch/ar58064/mitoxy/data/raw/gse159115_pub...
1,single_cell_annotation,/scratch/ar58064/mitoxy/data/raw/gse159115_pub...
2,spatial_tar,/scratch/ar58064/mitoxy/data/raw/gse210041_pub...
3,javelin_workbook,/scratch/ar58064/mitoxy/data/raw/javelin_renal...
4,cptac_rna,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...


Public-source integrity audit: PASS


In [2]:
def decode_array(values):
    values = np.asarray(values)
    if values.dtype.kind in "SO":
        return np.array([
            value.decode() if isinstance(value, bytes) else str(value)
            for value in values
        ])
    return values.astype(str)


def zscore_frame(frame):
    frame = frame.astype(float)
    standard_deviation = frame.std(axis=0, ddof=0).replace(0, np.nan)
    result = frame.subtract(frame.mean(axis=0), axis=1).divide(
        standard_deviation, axis=1
    )
    if result.isna().any().any():
        bad = result.columns[result.isna().any()].tolist()
        raise ValueError(f"Cannot standardize genes with invalid variance: {bad}")
    return result


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)
    adjusted = np.full(values.shape, np.nan)
    adjusted[valid] = multipletests(values[valid], method="fdr_bh")[1]
    return adjusted


def fisher_interval(rho, n, alpha_z=1.96):
    z = np.arctanh(np.clip(float(rho), -0.999999, 0.999999))
    standard_error = 1 / np.sqrt(int(n) - 3)
    return (
        float(np.tanh(z - alpha_z * standard_error)),
        float(np.tanh(z + alpha_z * standard_error)),
    )


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def find_column(columns, include_any, required=True):
    terms = [normalized(term) for term in include_any]
    matches = [
        column for column in columns
        if any(term in normalized(column) for term in terms)
    ]
    if not matches:
        if required:
            raise KeyError(
                f"No column matched {include_any}. Available: {list(columns)}"
            )
        return None
    return sorted(matches, key=lambda value: (len(str(value)), str(value)))[0]


def read_10x_h5(path):
    with h5py.File(path, "r") as handle:
        if "matrix" in handle:
            group = handle["matrix"]
        elif all(key in handle for key in ["data", "indices", "indptr", "shape"]):
            group = handle
        else:
            compatible = [
                handle[key] for key in handle.keys()
                if isinstance(handle[key], h5py.Group)
                and all(
                    field in handle[key]
                    for field in ["data", "indices", "indptr", "shape"]
                )
            ]
            if len(compatible) != 1:
                raise RuntimeError(
                    f"Could not uniquely identify the 10x matrix group in {path}"
                )
            group = compatible[0]
        data = group["data"][:]
        indices = group["indices"][:]
        indptr = group["indptr"][:]
        shape = tuple(group["shape"][:])
        matrix = sparse.csc_matrix((data, indices, indptr), shape=shape)
        barcodes = decode_array(group["barcodes"][:])
        if "features" in group:
            features = group["features"]
            name_key = "name" if "name" in features else "gene_names"
            genes = decode_array(features[name_key][:]).astype(str)
        else:
            name_key = "gene_names" if "gene_names" in group else "genes"
            genes = decode_array(group[name_key][:]).astype(str)
    return matrix, genes, barcodes


def collapse_duplicate_genes(matrix, genes):
    genes = pd.Index(pd.Series(genes).astype(str).str.upper())
    unique = genes.unique()
    if len(unique) == len(genes):
        return matrix, unique.to_numpy()
    mapping = pd.Series(np.arange(len(unique)), index=unique)
    rows = mapping.loc[genes].to_numpy()
    aggregator = sparse.csr_matrix(
        (np.ones(len(rows)), (rows, np.arange(len(rows)))),
        shape=(len(unique), len(rows)),
    )
    return aggregator @ matrix, unique.to_numpy()


def selected_log_normalized(matrix, genes, selected_genes):
    matrix, genes = collapse_duplicate_genes(matrix, genes)
    gene_index = pd.Series(np.arange(len(genes)), index=genes)
    present = [gene for gene in selected_genes if gene in gene_index.index]
    missing = sorted(set(selected_genes) - set(present))
    if missing:
        warnings.warn(f"Genes absent from matrix and omitted: {missing}")
    library_size = np.asarray(matrix.sum(axis=0)).ravel().astype(float)
    library_size[library_size == 0] = np.nan
    rows = matrix[gene_index.loc[present].to_numpy(), :].toarray().T
    normalized_values = np.log1p(rows / library_size[:, None] * 10_000)
    return pd.DataFrame(normalized_values, columns=present), rows


## Panel A — GSE159115 paired single-cell compartment localization

Detection percentages use raw UMI counts. Patient-level tests use mean
library-size-normalized, log-transformed expression within each compartment,
followed by paired Wilcoxon tests and Benjamini–Hochberg correction across the
prespecified gene family.


In [3]:
single_cell_extract = RAW["single_cell_tar"].parent / "extracted"
h5_files = sorted(single_cell_extract.rglob("*.h5"))

if not h5_files:
    raise FileNotFoundError(
        f"No extracted GSE159115 H5 files under {single_cell_extract}. "
        "Rerun the GEO extraction cell in 00_Paper_Data_Download.ipynb."
    )

annotation = pd.read_csv(
    RAW["single_cell_annotation"],
    compression="gzip",
)
annotation.columns = [
    str(column).strip()
    for column in annotation.columns
]


# ------------------------------------------------------------------
# Identify the annotation cell-barcode column.
# ------------------------------------------------------------------
cell_id_scores = {}

for column in annotation.columns:
    values = annotation[column].astype(str)

    cell_id_scores[column] = values.str.contains(
        r"[ACGT]{14,18}",
        regex=True,
        na=False,
    ).mean()

cell_id_column = max(
    cell_id_scores,
    key=cell_id_scores.get,
)

if cell_id_scores[cell_id_column] < 0.5:
    raise RuntimeError(
        "Could not identify the GSE159115 annotation "
        "cell-barcode column."
    )


# ------------------------------------------------------------------
# Identify the actual cell-type annotation column.
#
# A sample/disease column may contain 'ccRCC' in every row, so simply
# selecting the column with the most tumor-related matches can choose
# the wrong field. The correct column must contain both tumor-like and
# myeloid-like annotations.
# ------------------------------------------------------------------
tumor_pattern = (
    r"tumou?r|malignant|neoplastic|cancer[ _-]*cell|"
    r"ccrcc|epithelial"
)

myeloid_pattern = (
    r"myeloid|macro(?:phage)?|mono(?:cyte)?|"
    r"dendritic|(^|[ _/-])dc([ _/-]|$)|"
    r"neutro|granulocyte"
)

cell_type_candidates = []

for column in annotation.columns:
    values = (
        annotation[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    n_unique = values.nunique(dropna=True)

    tumor_fraction = values.str.contains(
        tumor_pattern,
        regex=True,
        na=False,
    ).mean()

    myeloid_fraction = values.str.contains(
        myeloid_pattern,
        regex=True,
        na=False,
    ).mean()

    column_name = str(column).lower()

    name_bonus = 0.0
    if any(
        term in column_name
        for term in [
            "cell_type",
            "cell type",
            "celltype",
            "annotation",
            "cluster",
            "identity",
            "ident",
        ]
    ):
        name_bonus = 1.0

    # Cell-type columns should contain multiple categories, but not
    # essentially one unique identifier per cell.
    diversity_ok = 2 <= n_unique <= 200

    if (
        diversity_ok
        and tumor_fraction > 0
        and myeloid_fraction > 0
    ):
        score = (
            name_bonus
            + tumor_fraction
            + myeloid_fraction
        )

        cell_type_candidates.append(
            {
                "column": column,
                "score": score,
                "n_unique": n_unique,
                "tumor_fraction": tumor_fraction,
                "myeloid_fraction": myeloid_fraction,
            }
        )

if not cell_type_candidates:
    annotation_summary = []

    for column in annotation.columns:
        values = annotation[column].dropna().astype(str)

        annotation_summary.append({
            "column": column,
            "n_unique": values.nunique(),
            "examples": values.unique()[:10].tolist(),
        })

    display(pd.DataFrame(annotation_summary))

    raise RuntimeError(
        "Could not identify an annotation column containing both "
        "tumor and myeloid cell labels. The annotation-column summary "
        "is displayed above."
    )

cell_type_audit = (
    pd.DataFrame(cell_type_candidates)
    .sort_values(
        ["score", "n_unique"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(cell_type_audit)

cell_type_column = cell_type_audit.loc[0, "column"]

print("Cell-ID column:", cell_id_column)
print("Cell-type column:", cell_type_column)

print("\nObserved cell-type labels:")
print(
    annotation[cell_type_column]
    .astype(str)
    .value_counts()
    .head(40)
)


# ------------------------------------------------------------------
# Extract the 10x barcode portion from the annotation identifier.
# ------------------------------------------------------------------
annotation_keys = annotation[cell_id_column].astype(str)

annotation_barcode = (
    annotation_keys
    .str.extract(
        r"([ACGT]{14,18}(?:-\d+)?)",
        expand=False,
    )
    .str.replace(
        r"-\d+$",
        "",
        regex=True,
    )
)


# ------------------------------------------------------------------
# Load and match each public GSE159115 10x matrix.
# ------------------------------------------------------------------
cell_blocks = []
sample_audit = []

for h5_path in h5_files:
    matrix, genes, barcodes = read_10x_h5(h5_path)

    values, raw_selected = selected_log_normalized(
        matrix,
        genes,
        SINGLE_CELL_FAMILY,
    )

    barcode_base = (
        pd.Series(barcodes, dtype=str)
        .str.replace(
            r"-\d+$",
            "",
            regex=True,
        )
    )

    # Most filenames contain both a GSM accession and an SI specimen
    # identifier. The annotation generally uses the SI identifier, so
    # prefer SI tokens when both are available.
    filename_tokens = re.findall(
        r"SI[_-]?\d+|GSM\d+",
        h5_path.name,
        flags=re.I,
    )

    filename_tokens = sorted(
        set(filename_tokens),
        key=lambda token: (
            0 if token.upper().startswith("SI") else 1,
            token,
        ),
    )

    candidate = annotation.loc[
        annotation_barcode.isin(
            set(barcode_base)
        )
    ].copy()

    matched_token = None

    for token in filename_tokens:
        token_mask = candidate.astype(str).apply(
            lambda column: column.str.contains(
                token,
                case=False,
                regex=False,
                na=False,
            )
        ).any(axis=1)

        if token_mask.any():
            candidate = candidate.loc[token_mask].copy()
            matched_token = token
            break

    if matched_token is None:
        # This fallback is only safe when the barcode is unique within
        # the annotation table.
        barcode_frequency = annotation_barcode.value_counts()

        unique_barcodes = set(
            barcode_frequency.loc[
                barcode_frequency.eq(1)
            ].index
        )

        candidate = candidate.loc[
            annotation_barcode.loc[
                candidate.index
            ].isin(unique_barcodes)
        ].copy()

    if candidate.empty:
        warnings.warn(
            f"No annotation overlap for {h5_path.name}; skipping."
        )
        continue

    candidate["_barcode_base"] = (
        annotation_barcode.loc[candidate.index]
    )

    candidate = candidate.drop_duplicates(
        "_barcode_base"
    )

    metadata = pd.DataFrame({
        "_barcode_base": barcode_base,
        "_matrix_row": np.arange(
            len(barcodes)
        ),
    }).merge(
        candidate,
        on="_barcode_base",
        how="inner",
        validate="one_to_one",
    )

    if metadata.empty:
        warnings.warn(
            f"No uniquely matched cells for {h5_path.name}; skipping."
        )
        continue

    matrix_rows = metadata[
        "_matrix_row"
    ].to_numpy(dtype=int)

    block = (
        values
        .iloc[matrix_rows]
        .reset_index(drop=True)
    )

    raw_block = raw_selected[
        matrix_rows,
        :
    ]

    for gene_index, gene in enumerate(values.columns):
        block[f"detected_{gene}"] = (
            raw_block[:, gene_index] > 0
        )

    patient_label = (
        matched_token
        if matched_token is not None
        else h5_path.stem
    )

    block["patient"] = patient_label

    block["cell_type"] = (
        metadata[cell_type_column]
        .astype(str)
        .str.strip()
        .to_numpy()
    )

    cell_blocks.append(block)

    sample_audit.append({
        "file": h5_path.name,
        "patient": patient_label,
        "matrix_cells": len(barcodes),
        "annotated_cells": len(block),
    })

if not cell_blocks:
    raise RuntimeError(
        "No annotated GSE159115 ccRCC cells were reconstructed."
    )

display(pd.DataFrame(sample_audit))


# ------------------------------------------------------------------
# Collapse the public cell annotations into the two prespecified
# compartments.
# ------------------------------------------------------------------
single_cells = pd.concat(
    cell_blocks,
    ignore_index=True,
)

type_text = (
    single_cells["cell_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

single_cells["compartment"] = np.select(
    [
        type_text.str.contains(
            myeloid_pattern,
            regex=True,
            na=False,
        ),
        type_text.str.contains(
            tumor_pattern,
            regex=True,
            na=False,
        ),
    ],
    [
        "Myeloid",
        "Tumor",
    ],
    default="Other",
)

print("\nCollapsed compartment counts:")
print(
    single_cells["compartment"]
    .value_counts(dropna=False)
)

print("\nOriginal labels classified as myeloid:")
print(
    single_cells.loc[
        single_cells["compartment"].eq("Myeloid"),
        "cell_type",
    ]
    .value_counts()
    .head(30)
)

print("\nOriginal labels classified as tumor:")
print(
    single_cells.loc[
        single_cells["compartment"].eq("Tumor"),
        "cell_type",
    ]
    .value_counts()
    .head(30)
)

analysis_cells = single_cells.loc[
    single_cells["compartment"].isin(
        ["Tumor", "Myeloid"]
    )
].copy()

observed_compartments = set(
    analysis_cells["compartment"]
)

required_compartments = {
    "Tumor",
    "Myeloid",
}

missing_compartments = sorted(
    required_compartments
    - observed_compartments
)

if missing_compartments:
    raise RuntimeError(
        "The reconstructed single-cell data are missing the "
        f"following compartments: {missing_compartments}. "
        "Review the displayed original cell-type labels."
    )


# ------------------------------------------------------------------
# Patient-paired expression tests.
# ------------------------------------------------------------------
paired_rows = []

for gene in SINGLE_CELL_FAMILY:
    if gene not in analysis_cells.columns:
        warnings.warn(
            f"{gene} is absent from the reconstructed expression table."
        )
        continue

    patient_compartment = analysis_cells.pivot_table(
        index="patient",
        columns="compartment",
        values=gene,
        aggfunc="mean",
    )

    if not required_compartments.issubset(
        patient_compartment.columns
    ):
        raise RuntimeError(
            f"{gene}: paired table contains compartments "
            f"{patient_compartment.columns.tolist()}"
        )

    patient_compartment = patient_compartment.dropna(
        subset=["Tumor", "Myeloid"]
    )

    if len(patient_compartment) < 3:
        raise RuntimeError(
            f"Too few paired patients for {gene}: "
            f"{len(patient_compartment)}"
        )

    statistic, p_value = wilcoxon(
        patient_compartment["Tumor"],
        patient_compartment["Myeloid"],
        alternative="two-sided",
        zero_method="wilcox",
    )

    paired_rows.append({
        "gene": gene,
        "n_patients": len(
            patient_compartment
        ),
        "paired_p": float(p_value),
        "tumor_mean": float(
            patient_compartment["Tumor"].mean()
        ),
        "myeloid_mean": float(
            patient_compartment["Myeloid"].mean()
        ),
    })

paired = pd.DataFrame(paired_rows)

paired["paired_fdr"] = bh_adjust(
    paired["paired_p"]
)


# ------------------------------------------------------------------
# Detection percentages used in Figure 5A.
# ------------------------------------------------------------------
localization_rows = []

for gene in [
    "MYD88",
    "TNF",
    "TLR2",
    "TLR4",
    "RICTOR",
]:
    detection_column = f"detected_{gene}"

    if detection_column not in analysis_cells.columns:
        raise KeyError(
            f"Missing detection column: {detection_column}"
        )

    tumor_detection = analysis_cells.loc[
        analysis_cells["compartment"].eq("Tumor"),
        detection_column,
    ]

    myeloid_detection = analysis_cells.loc[
        analysis_cells["compartment"].eq("Myeloid"),
        detection_column,
    ]

    localization_rows.append({
        "gene": gene,
        "tumor_detection_pct": (
            100 * tumor_detection.mean()
        ),
        "myeloid_detection_pct": (
            100 * myeloid_detection.mean()
        ),
    })

panel_a = pd.DataFrame(
    localization_rows
).merge(
    paired[
        [
            "gene",
            "n_patients",
            "paired_p",
            "paired_fdr",
        ]
    ],
    on="gene",
    how="left",
    validate="one_to_one",
)

if panel_a.isna().any().any():
    missing_summary = panel_a.loc[
        panel_a.isna().any(axis=1)
    ]

    display(missing_summary)

    raise ValueError(
        "Panel A contains missing reconstructed values."
    )

panel_a.to_csv(
    OUTPUT_DIR
    / "panel_a_compartment_localization.csv",
    index=False,
)

print("\nGSE159115 reconstruction summary")
print("--------------------------------")
print("Annotated ccRCC cells:", len(single_cells))
print(
    "Tumor/myeloid cells retained:",
    len(analysis_cells),
)
print(
    "Paired patients:",
    paired["n_patients"].min(),
)

display(panel_a)

/tmp/ipykernel_1007968/599745978.py:83: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  myeloid_fraction = values.str.contains(
/tmp/ipykernel_1007968/599745978.py:83: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  myeloid_fraction = values.str.contains(
/tmp/ipykernel_1007968/599745978.py:83: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  myeloid_fraction = values.str.contains(
/tmp/ipykernel_1007968/599745978.py:83: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  myeloid_fraction = values.str.contains(
/tmp/ipykernel_1007968/599745978.py:83: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually 

,column,score,n_unique,tumor_fraction,myeloid_fraction
0,anno,0.642231,13,0.440476,0.201754
1,label,0.642231,14,0.440476,0.201754


Cell-ID column: cell
Cell-type column: anno

Observed cell-type labels:
anno
Tumor          9139
Macro          4149
Endo_PLVAP     2573
vSMC           2119
Tcell           960
Peri            589
Endo_ACKR1      487
ua              239
Tcell_CD8       207
Plasma          116
Bcell            69
Mast             64
Macro_MKI67      37
Name: count, dtype: int64


,file,patient,matrix_cells,annotated_cells
0,GSM4819725_SI_18854_filtered_gene_bc_matrices_...,SI_18854,1704,1665
1,GSM4819726_SI_18856_filtered_gene_bc_matrices_...,GSM4819726_SI_18856_filtered_gene_bc_matrices_h5,1410,44
2,GSM4819727_SI_18855_filtered_gene_bc_matrices_...,SI_18855,1660,1362
3,GSM4819728_SI_19704_filtered_gene_bc_matrices_...,GSM4819728_SI_19704_filtered_gene_bc_matrices_h5,923,31
4,GSM4819729_SI_19703_filtered_gene_bc_matrices_...,SI_19703,3674,3331
5,GSM4819730_SI_21255_filtered_gene_bc_matrices_...,GSM4819730_SI_21255_filtered_gene_bc_matrices_h5,1097,32
6,GSM4819731_SI_21256_filtered_gene_bc_matrices_...,GSM4819731_SI_21256_filtered_gene_bc_matrices_h5,578,18
7,GSM4819732_SI_21561_filtered_gene_bc_matrices_...,GSM4819732_SI_21561_filtered_gene_bc_matrices_h5,2853,76
8,GSM4819733_SI_22369_filtered_gene_bc_matrices_...,GSM4819733_SI_22369_filtered_gene_bc_matrices_h5,1644,52
9,GSM4819734_SI_22368_filtered_gene_bc_matrices_...,SI_22368,2872,2808


/tmp/ipykernel_1007968/599745978.py:378: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  type_text.str.contains(



Collapsed compartment counts:
compartment
Tumor      9279
Other      7540
Myeloid    4239
Name: count, dtype: int64

Original labels classified as myeloid:
cell_type
Macro          4202
Macro_MKI67      37
Name: count, dtype: int64

Original labels classified as tumor:
cell_type
Tumor    9279
Name: count, dtype: int64

GSE159115 reconstruction summary
--------------------------------
Annotated ccRCC cells: 21058
Tumor/myeloid cells retained: 13518
Paired patients: 14


,gene,tumor_detection_pct,myeloid_detection_pct,n_patients,paired_p,paired_fdr
0,MYD88,7.953443,19.792404,14,0.099481,0.182382
1,TNF,2.575709,26.492097,14,0.091161,0.182382
2,TLR2,3.448647,39.773531,14,0.022909,0.129354
3,TLR4,1.207027,15.876386,14,0.028417,0.129354
4,RICTOR,25.336782,7.525360,14,0.035278,0.129354


## Panels B and C — JAVELIN/CPTAC coupling and covariate adjustment

The public JAVELIN workbook labels its clinical and normalized-expression
tables as Supplementary Tables 11 and 13. The helper below permits harmless
header offsets in different workbook releases but requires the component
genes and public sample identifiers to determine the orientation.


In [7]:
# ================================================================
# Resolve JAVELIN clinical adjustment fields
#
# Public S11 abbreviations:
#   TRT01P       = randomized treatment
#   PDL1FL       = PD-L1 status
#   TCGA_cluster = TCGA molecular cluster
# ================================================================

def resolve_clinical_column(
    columns,
    exact_aliases=(),
    contains_aliases=(),
    required=True,
):
    exact_lookup = {
        str(column).strip().casefold(): column
        for column in columns
    }

    for alias in exact_aliases:
        key = str(alias).strip().casefold()

        if key in exact_lookup:
            return exact_lookup[key]

    normalized_aliases = [
        normalized(alias)
        for alias in contains_aliases
    ]

    matches = []

    for column in columns:
        column_name = normalized(column)

        if any(
            alias in column_name
            for alias in normalized_aliases
        ):
            matches.append(column)

    if matches:
        return sorted(
            matches,
            key=lambda value: (
                len(str(value)),
                str(value),
            ),
        )[0]

    if required:
        raise KeyError(
            "Could not resolve clinical column. "
            f"Exact aliases={list(exact_aliases)}; "
            f"contains aliases={list(contains_aliases)}; "
            f"available columns={list(columns)}"
        )

    return None


age_column = resolve_clinical_column(
    javelin_clinical.columns,
    exact_aliases=[
        "AGE",
    ],
    contains_aliases=[
        "age",
    ],
)

sex_column = resolve_clinical_column(
    javelin_clinical.columns,
    exact_aliases=[
        "SEX",
    ],
    contains_aliases=[
        "sex",
        "gender",
    ],
)

treatment_column = resolve_clinical_column(
    javelin_clinical.columns,
    exact_aliases=[
        "TRT01P",
    ],
    contains_aliases=[
        "treatment",
        "randomized treatment",
        "treatment arm",
        "randomized arm",
    ],
)

pdl1_column = resolve_clinical_column(
    javelin_clinical.columns,
    exact_aliases=[
        "PDL1FL",
    ],
    contains_aliases=[
        "pd l1",
        "pdl1",
    ],
)

cluster_column = resolve_clinical_column(
    javelin_clinical.columns,
    exact_aliases=[
        "TCGA_cluster",
    ],
    contains_aliases=[
        "tcga cluster",
        "molecular cluster",
    ],
    required=False,
)


print("Resolved JAVELIN adjustment fields")
print("-----------------------------------")
print("Age:          ", age_column)
print("Sex:          ", sex_column)
print("Treatment:    ", treatment_column)
print("PD-L1:        ", pdl1_column)
print("TCGA cluster: ", cluster_column)


print("\nClinical-field completeness among matched samples")
print("--------------------------------------------------")

clinical_field_audit = pd.DataFrame({
    "field": [
        "Age",
        "Sex",
        "Treatment",
        "PD-L1",
        "TCGA cluster",
    ],
    "column": [
        age_column,
        sex_column,
        treatment_column,
        pdl1_column,
        cluster_column,
    ],
})

clinical_field_audit["nonmissing"] = [
    (
        javelin[column]
        .notna()
        .sum()
        if column is not None
        else 0
    )
    for column in clinical_field_audit["column"]
]

clinical_field_audit["unique_values"] = [
    (
        javelin[column]
        .nunique(dropna=True)
        if column is not None
        else 0
    )
    for column in clinical_field_audit["column"]
]

display(clinical_field_audit)


print("\nCategorical value audit")

for label, column in [
    ("Sex", sex_column),
    ("Treatment", treatment_column),
    ("PD-L1", pdl1_column),
    ("TCGA cluster", cluster_column),
]:
    if column is not None:
        print(f"\n{label} ({column})")

        print(
            javelin[column]
            .value_counts(dropna=False)
        )

Resolved JAVELIN adjustment fields
-----------------------------------
Age:           AGE
Sex:           SEX
Treatment:     TRT01P
PD-L1:         PDL1FL
TCGA cluster:  TCGA_cluster

Clinical-field completeness among matched samples
--------------------------------------------------


,field,column,nonmissing,unique_values
0,Age,AGE,726,56
1,Sex,SEX,726,2
2,Treatment,TRT01P,726,2
3,PD-L1,PDL1FL,726,2
4,TCGA cluster,TCGA_cluster,723,5



Categorical value audit

Sex (SEX)
SEX
M    548
F    178
Name: count, dtype: int64

Treatment (TRT01P)
TRT01P
Sunitinib            372
Avelumab+Axitinib    354
Name: count, dtype: int64

PD-L1 (PDL1FL)
PDL1FL
Y    516
N    210
Name: count, dtype: int64

TCGA cluster (TCGA_cluster)
TCGA_cluster
m3       231
m1       179
m4       155
m2       143
Other     15
NaN        3
Name: count, dtype: int64


In [8]:
def read_gene_matrix(path, genes):
    raw = pd.read_csv(path, sep="\t", index_col=0)
    raw.index = raw.index.astype(str).str.strip().str.upper()
    raw.columns = raw.columns.astype(str).str.strip()
    if set(genes).issubset(raw.index):
        return raw.loc[genes].T.apply(pd.to_numeric, errors="coerce")
    if set(genes).issubset({column.upper() for column in raw.columns}):
        raw.columns = raw.columns.str.upper()
        return raw[genes].apply(pd.to_numeric, errors="coerce")
    raise KeyError(f"Could not orient {path}; required genes not found")


cptac_expression = read_gene_matrix(
    RAW["cptac_rna"], MITOXY_GENES + MYELOID_GENES
).dropna()
cptac_z = zscore_frame(cptac_expression)
cptac_mitoxy = cptac_z[MITOXY_GENES].mean(axis=1)
cptac_myeloid = cptac_z[MYELOID_GENES].mean(axis=1)

javelin_complete = javelin[["mitoxy", "myeloid"]].dropna()
javelin_rho, javelin_p = spearmanr(
    javelin_complete["mitoxy"], javelin_complete["myeloid"]
)
cptac_rho, cptac_p = spearmanr(cptac_mitoxy, cptac_myeloid)

cohorts = pd.DataFrame([
    {
        "cohort": "JAVELIN101", "n": len(javelin_complete),
        "rho": float(javelin_rho), "p_value": float(javelin_p),
    },
    {
        "cohort": "CPTAC", "n": len(cptac_expression),
        "rho": float(cptac_rho), "p_value": float(cptac_p),
    },
])
cohorts[["ci_low", "ci_high"]] = cohorts.apply(
    lambda row: pd.Series(fisher_interval(row["rho"], row["n"])), axis=1
)

fisher_z = np.arctanh(cohorts["rho"].to_numpy())
weights = cohorts["n"].to_numpy() - 3
pooled_z = np.average(fisher_z, weights=weights)
pooled_standard_error = 1 / np.sqrt(weights.sum())
meta = pd.DataFrame([{
    "cohort": "Fixed-effect summary",
    "n": int(cohorts["n"].sum()),
    "rho": float(np.tanh(pooled_z)),
    "p_value": np.nan,
    "ci_low": float(np.tanh(pooled_z - 1.96 * pooled_standard_error)),
    "ci_high": float(np.tanh(pooled_z + 1.96 * pooled_standard_error)),
}])
panel_b = pd.concat([cohorts, meta], ignore_index=True)
panel_b.to_csv(OUTPUT_DIR / "panel_b_cross_cohort_coupling.csv", index=False)
display(panel_b)


,cohort,n,rho,p_value,ci_low,ci_high
0,JAVELIN101,726,0.448136,3.752553e-37,0.388025,0.504451
1,CPTAC,110,0.329622,4.374167e-04,0.151742,0.486820
2,Fixed-effect summary,836,0.433600,NaN,0.376770,0.487179


In [9]:
def partial_rank_correlation(data, x, y, numeric_covariates, categorical_covariates):
    fields = [x, y, *numeric_covariates, *categorical_covariates]
    complete = data[fields].replace([np.inf, -np.inf], np.nan).dropna().copy()
    ranked = complete[[x, y, *numeric_covariates]].rank(method="average")
    design_parts = [ranked[numeric_covariates]] if numeric_covariates else []
    if categorical_covariates:
        design_parts.append(pd.get_dummies(
            complete[categorical_covariates].astype(str),
            drop_first=True, dtype=float,
        ))
    design = pd.concat(design_parts, axis=1)
    design = sm.add_constant(design, has_constant="add").astype(float)
    x_residual = sm.OLS(ranked[x].astype(float), design).fit().resid
    y_residual = sm.OLS(ranked[y].astype(float), design).fit().resid
    rho, p_value = pearsonr(x_residual, y_residual)
    return len(complete), float(rho), float(p_value)


raw_complete = javelin[["mitoxy", "myeloid"]].dropna()
raw_rho, raw_p = spearmanr(raw_complete["mitoxy"], raw_complete["myeloid"])

adjustment = javelin.copy()
adjustment["age_numeric"] = pd.to_numeric(
    adjustment[age_column], errors="coerce"
)
categorical_covariates = [sex_column, treatment_column, pdl1_column]
if cluster_column is not None:
    categorical_covariates.append(cluster_column)

adjusted_n, adjusted_rho, adjusted_p = partial_rank_correlation(
    adjustment,
    x="mitoxy", y="myeloid",
    numeric_covariates=["age_numeric", "angio"],
    categorical_covariates=categorical_covariates,
)

panel_c = pd.DataFrame([
    {
        "model": "Raw", "n": len(raw_complete),
        "rho": float(raw_rho), "p_value": float(raw_p),
    },
    {
        "model": "Adjusted partial", "n": adjusted_n,
        "rho": adjusted_rho, "p_value": adjusted_p,
    },
])
panel_c[["ci_low", "ci_high"]] = panel_c.apply(
    lambda row: pd.Series(fisher_interval(row["rho"], row["n"])), axis=1
)
panel_c.to_csv(OUTPUT_DIR / "panel_c_adjusted_coupling.csv", index=False)
display(panel_c)
print("Adjustment covariates:", ["age", "angiogenesis", *categorical_covariates])


,model,n,rho,p_value,ci_low,ci_high
0,Raw,726,0.448136,3.752553e-37,0.388025,0.504451
1,Adjusted partial,723,0.379236,3.802894e-26,0.315032,0.439985


Adjustment covariates: ['age', 'angiogenesis', 'SEX', 'TRT01P', 'PDL1FL', 'TCGA_cluster']


## Panel D — GSE210041 frozen compartment-aware spatial analysis

The same analysis is applied to both tumors. Counts are normalized to 10,000
UMIs per spot, log transformed, and standardized by gene. Malignant-enriched
spots are the upper tumor-marker quartile below the median myeloid-marker
score; myeloid-enriched spots use the reciprocal rule. For every malignant
spot, the mean inflammatory score of its six nearest myeloid spots is tested
against Mitoxy. The permutation test shuffles inflammatory scores within the
fixed myeloid compartment while retaining all coordinates and class labels.


In [10]:
spatial_extract = RAW["spatial_tar"].parent / "extracted"
spatial_h5 = sorted(spatial_extract.rglob("*filtered_feature_bc_matrix.h5"))
spatial_archives = sorted(spatial_extract.rglob("*spatial.tar.gz"))
if len(spatial_h5) < 2 or len(spatial_archives) < 2:
    raise FileNotFoundError(
        f"Expected two filtered H5 matrices and two spatial archives under "
        f"{spatial_extract}; found {len(spatial_h5)} and {len(spatial_archives)}"
    )


def sample_label(path):
    name = path.name.upper()
    if "GSM6415705" in name or "GLMF1" in name:
        return "T1"
    if "GSM6415706" in name or "GLMF2" in name:
        return "T7"
    raise ValueError(f"Unrecognized GSE210041 sample file: {path.name}")


def spatial_positions(archive_path):
    with tarfile.open(archive_path, "r:gz") as archive:
        candidates = [
            member for member in archive.getmembers()
            if "tissue_positions" in Path(member.name).name
        ]
        if len(candidates) != 1:
            raise RuntimeError(
                f"Expected one tissue-position file in {archive_path}; "
                f"found {[member.name for member in candidates]}"
            )
        handle = archive.extractfile(candidates[0])
        if handle is None:
            raise RuntimeError(f"Could not read {candidates[0].name}")
        if Path(candidates[0].name).name == "tissue_positions_list.csv":
            table = pd.read_csv(handle, header=None)
        else:
            table = pd.read_csv(handle)
    if table.shape[1] >= 6 and str(table.columns[0]).isdigit():
        table = table.iloc[:, :6]
        table.columns = [
            "barcode", "in_tissue", "array_row", "array_col",
            "pxl_row", "pxl_col",
        ]
    else:
        rename = {
            "pxl_row_in_fullres": "pxl_row",
            "pxl_col_in_fullres": "pxl_col",
        }
        table = table.rename(columns=rename)
    table["barcode"] = table["barcode"].astype(str)
    return table


def spatial_sample(matrix_path, positions_path, permutations=5000, seed=20260913):
    all_genes = list(dict.fromkeys(
        MITOXY_GENES + MYELOID_GENES + TUMOR_MARKERS + MYELOID_MARKERS
    ))
    matrix, genes, barcodes = read_10x_h5(matrix_path)
    values, _ = selected_log_normalized(matrix, genes, all_genes)
    values.index = pd.Index(barcodes.astype(str), name="barcode")
    positions = spatial_positions(positions_path)
    positions = positions.loc[pd.to_numeric(
        positions["in_tissue"], errors="coerce"
    ).eq(1)].copy()
    spot = positions.merge(
        values.reset_index(), on="barcode", how="inner", validate="one_to_one"
    )
    if len(spot) < 500:
        raise RuntimeError(
            f"Unexpectedly few in-tissue spots for {matrix_path.name}: {len(spot)}"
        )

    available_targets = [gene for gene in MITOXY_GENES + MYELOID_GENES if gene in spot]
    available_tumor = [gene for gene in TUMOR_MARKERS if gene in spot]
    available_myeloid = [gene for gene in MYELOID_MARKERS if gene in spot]
    if len(available_targets) != 8:
        raise KeyError(f"Spatial target genes missing: {available_targets}")
    if len(available_tumor) < 4 or len(available_myeloid) < 4:
        raise KeyError(
            f"Insufficient compartment markers: tumor={available_tumor}, "
            f"myeloid={available_myeloid}"
        )

    gene_z = zscore_frame(spot[available_targets + available_tumor + available_myeloid])
    spot["mitoxy"] = gene_z[MITOXY_GENES].mean(axis=1)
    spot["myeloid4"] = gene_z[MYELOID_GENES].mean(axis=1)
    spot["tumor_marker_score"] = gene_z[available_tumor].mean(axis=1)
    spot["myeloid_marker_score"] = gene_z[available_myeloid].mean(axis=1)
    spot["malignant_enriched"] = (
        spot["tumor_marker_score"].gt(spot["tumor_marker_score"].quantile(0.75))
        & spot["myeloid_marker_score"].lt(spot["myeloid_marker_score"].median())
    )
    spot["myeloid_enriched"] = (
        spot["myeloid_marker_score"].gt(spot["myeloid_marker_score"].quantile(0.75))
        & spot["tumor_marker_score"].lt(spot["tumor_marker_score"].median())
    )

    malignant = spot.loc[spot["malignant_enriched"]].copy()
    myeloid = spot.loc[spot["myeloid_enriched"]].copy()
    coordinate_columns = ["pxl_row", "pxl_col"]
    malignant_coordinates = malignant[coordinate_columns].astype(float).to_numpy()
    myeloid_coordinates = myeloid[coordinate_columns].astype(float).to_numpy()
    nearest = cKDTree(myeloid_coordinates).query(
        malignant_coordinates, k=min(6, len(myeloid))
    )[1]
    if nearest.ndim == 1:
        nearest = nearest[:, None]
    myeloid_scores = myeloid["myeloid4"].to_numpy()
    neighbor_score = myeloid_scores[nearest].mean(axis=1)
    rho, nominal_p = spearmanr(malignant["mitoxy"], neighbor_score)

    rng = np.random.default_rng(seed)
    null = np.empty(permutations)
    malignant_score = malignant["mitoxy"].to_numpy()
    for iteration in range(permutations):
        shuffled = rng.permutation(myeloid_scores)
        null[iteration] = spearmanr(
            malignant_score, shuffled[nearest].mean(axis=1)
        ).statistic
    permutation_p = (
        1 + np.sum(np.abs(null) >= abs(rho))
    ) / (permutations + 1)
    return {
        "n_malignant": len(malignant),
        "n_myeloid": len(myeloid),
        "neighbor_rho": float(rho),
        "nominal_p": float(nominal_p),
        "permutation_p": float(permutation_p),
    }


h5_by_sample = {sample_label(path): path for path in spatial_h5}
spatial_by_sample = {sample_label(path): path for path in spatial_archives}
if set(h5_by_sample) != {"T1", "T7"} or set(spatial_by_sample) != {"T1", "T7"}:
    raise RuntimeError(
        f"Could not resolve both spatial samples: "
        f"H5={h5_by_sample}, spatial={spatial_by_sample}"
    )

spatial_rows = []
for index, sample in enumerate(["T1", "T7"]):
    result = spatial_sample(
        h5_by_sample[sample], spatial_by_sample[sample],
        permutations=5000, seed=20260913 + index,
    )
    spatial_rows.append({"sample": sample, **result})

panel_d = pd.DataFrame(spatial_rows)
panel_d.to_csv(OUTPUT_DIR / "panel_d_spatial_replication.csv", index=False)
display(panel_d)

print("\nFigure 5 plotting tables saved to:")
print(OUTPUT_DIR)


,sample,n_malignant,n_myeloid,neighbor_rho,nominal_p,permutation_p
0,T1,463,478,0.139427,0.002641,0.059388
1,T7,556,621,0.035546,0.402847,0.467307



Figure 5 plotting tables saved to:
/scratch/ar58064/mitoxy/data/processed/figure_5_clean


## Output contract

The unchanged plotting notebook reads these files:

- `panel_a_compartment_localization.csv`
- `panel_b_cross_cohort_coupling.csv`
- `panel_c_adjusted_coupling.csv`
- `panel_d_spatial_replication.csv`
